# 03 Evaluation V2 - Simple, RAM-safe, Polars-first

Muc tieu:
- Code ngan gon, de hieu.
- Khong load full train/test vao RAM.
- Dung Polars + doc theo batch (pyarrow).
- Xuat artifact ro rang: predictions_topk.parquet, metrics_by_segment.parquet, metrics_summary.json.
- Theo chuan SingleResponse cho JSON summary.

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    BASE_PATH = '/content/drive/MyDrive/Thesis/'
    print(f"Running on Colab. Base Path: {BASE_PATH}")
else:
    BASE_PATH = './'
    print(f"Running on local device. Base Path: {BASE_PATH}")


DATA_DIR = os.path.join(BASE_PATH, 'data/processed/eval/')
MODEL_DIR = os.path.join(BASE_PATH, 'models/')
INDEX_PATH = os.path.join(BASE_PATH, 'data/processed/cb_sbert.index')
MAPPING_PATH = os.path.join(BASE_PATH, 'data/processed/item_mapping.csv')

if BASE_PATH not in sys.path:
    sys.path.append(BASE_PATH)

In [1]:
# ==========================================
# CELL 1: SETUP & IMPORTS
# ==========================================
import sys
import gc
import logging
from pathlib import Path

import numpy as np
import polars as pl
from tqdm.auto import tqdm
from ranx import Qrels, Run, evaluate

sys.path.append("..")

from app.recommenders.types import RecommendRequest
from app.recommenders.faiss_model import FaissContentBasedRecommender


logging.basicConfig(level=logging.INFO,force = True, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger("Evaluator")

K_RECOMMEND = 10
TOP_N_INTERNAL = 100
CHUNK_SIZE = 5000

Mounted at /content/drive
Drive mounted


In [2]:
# Cell 3 - Optional install
INSTALL_DEPS = False
if INSTALL_DEPS:
    if IN_COLAB:
        %pip install -q polars pyarrow numpy tqdm faiss-gpu-cu12
    else:
        %pip install -q polars pyarrow numpy tqdm faiss-cpu

In [3]:
# ==========================================
# CELL 2: LOAD DATA & PREPARE QRELS
# ==========================================
logger.info("Data Loading...")

# 1. Read Train/Test
train_df = pl.read_parquet("data/processed/eval/train_interactions.parquet")
test_df = pl.read_parquet("data/processed/eval/test_interactions.parquet")

# 2. Building  (Masking) from Train
logger.info("filtering User (Train Set)...")
train_history = (
    train_df.group_by("user_id")
    .agg(pl.col("book_id").cast(pl.Utf8))
    .to_dict(as_series=False)
)
seen_dict = {u: set(items) for u, items in zip(train_history["user_id"], train_history["book_id"])}

# 3. Build Ground Truth (Qrels)
logger.info("Buiding Qrels (Test Set) for ranx...")
test_history = (
    test_df.group_by("user_id")
    .agg(pl.col("book_id").cast(pl.Utf8))
    .to_dict(as_series=False)
)
# format: {"user1": {"itemA": 1, "itemB": 1}}
qrels_dict = {
    u: {item: 1 for item in items}
    for u, items in zip(test_history["user_id"], test_history["book_id"])
}
qrels = Qrels(qrels_dict)

# 4. (Giả lập) Load Vector Sở thích của User từ ổ cứng
# Lưu ý: Thay đường dẫn bằng file ma trận User Profile thực tế của bạn
logger.info("Loading User Profiles (MMAP)...")
try:
    # Ví dụ: file chứa vector của từng user đã được tính trước
    user_profiles = np.load("data/processed/eval/user_profiles.npy", mmap_mode='r')
    # Ánh xạ từ user_id sang index của ma trận (Cần có 1 dict ánh xạ)
    # user_id_to_idx = ...
except Exception as e:
    logger.warning("Chưa có sẵn User Profiles. Vui lòng đảm bảo bạn đã tính Vector cho User.")

test_users = list(qrels_dict.keys())
logger.info(f"Hoàn tất! Sẵn sàng đánh giá {len(test_users)} users.")

2026-04-14 06:44:41,993 | INFO | Config ready | base_dir=/content/drive/My Drive/Thesis/book_recsys | device=gpu | use_gpu=True


In [4]:
# ==========================================
# CELL 3: INIT MODEL
# ==========================================
logger.info("Khởi tạo FaissContentBasedRecommender...")

model = FaissContentBasedRecommender(
    index_path="data/processed/cb_sbert.index",
    item_mapping_path="data/processed/item_mapping.csv",
    use_gpu=True, # BẬT TÍNH NĂNG GPU LÊN
    gpu_device_id=0
)

# Nạp model (Tự động dùng MMAP để bảo vệ RAM)
model.load()

# Test hệ thống (Ping)
health = model.healthcheck()
logger.info(f"Trạng thái Model: {health}")

Run dir: /content/drive/My Drive/Thesis/book_recsys/artifacts/evaluation_v2/eval_v2_20260414_064442


In [5]:
# ==========================================
# CELL 4: CHUNKING EVALUATION LOOP
# ==========================================
logger.info(f"Bắt đầu chạy luồng dự đoán (Chunk size: {CHUNK_SIZE})...")

run_dict = {}

# Chạy vòng lặp chia lô (Chunking)
for i in tqdm(range(0, len(test_users), CHUNK_SIZE), desc="Đánh giá Batch"):
    batch_users = test_users[i : i + CHUNK_SIZE]
    batch_requests = []

    # 1. Đóng gói Request cho lô này
    for user_id in batch_users:
        # Lấy vector (Giả định bạn đã có hàm hoặc mapping để lấy vector của user)
        # user_idx = user_id_to_idx[user_id]
        # u_vec = user_profiles[user_idx]

        # DEMO MOCK VECTOR (Xóa dòng này khi ráp code thật)
        u_vec = np.random.rand(384).astype(np.float32)

        req = RecommendRequest(
            user_id=user_id,
            seen_item_ids=seen_dict.get(user_id, set()), # Trích xuất sách đã đọc để Filter
            user_vector=u_vec
        )
        batch_requests.append(req)

    # 2. Bắn sang GPU xử lý bằng True Batching (Nhanh như chớp)
    batch_results = model.recommend_batch(
        requests=batch_requests,
        k=K_RECOMMEND,
        topn=TOP_N_INTERNAL
    )

    # 3. Format kết quả về chuẩn của ranx
    for user_id, rec_items in batch_results.items():
        run_dict[user_id] = {item.item_id: item.score for item in rec_items}

    # 4. Ép dọn rác thủ công (Bảo vệ RAM 100%)
    del batch_requests, batch_results
    gc.collect()

# ==========================================
# TÍNH TOÁN METRICS CUỐI CÙNG
# ==========================================
logger.info("Hoàn tất dự đoán. Đang tính điểm Metrics...")
run = Run(run_dict)

# Khai báo các bài thi (Metrics)
metrics_to_compute = [f"hit_rate@{K_RECOMMEND}", f"recall@{K_RECOMMEND}", f"ndcg@{K_RECOMMEND}", "mrr"]

# Chấm điểm (Sử dụng Multi-threading của ranx)
report = evaluate(qrels, run, metrics_to_compute, threads=0)

# Hiển thị báo cáo tuyệt đẹp
print("\n" + "="*50)
print(f"🏆 BÁO CÁO KẾT QUẢ ĐÁNH GIÁ (MODEL: {model.model_name})")
print("="*50)
for metric, score in report.items():
    print(f" ➤ {metric.upper():<15}: {score:.4f}")
print("="*50)

2026-04-14 06:44:48,910 | INFO | NumExpr defaulting to 2 threads.
